# Using ICA to reduce artefacts

This section shows how to use independent component analysis (ICA) to reduce artefacts in the DAD MEG recordings. The workflow follows the structure of the FLUX ICA notebook, but the paths and artefact-review information are adapted to the local non-BIDS DAD data.

ICA is applied after MaxFilter and artefact annotation. The annotated files still contain the full continuous signal, but they also contain `BAD_blink`, `BAD_eye_movement`, and `BAD_muscle` time intervals from the previous notebook. ICA fitting can use those annotations to avoid fitting components from already-marked artefact segments.

The central decision in this notebook is manual: inspect the ICA source time courses and component topographies, decide which components represent artefacts, and list them in `ica.exclude`. In this pipeline, those component indices come from `ica.exclude_components` in `config.toml`. Leave that list empty until the components have been reviewed.

## Preparation

Import the relevant modules.

In [1]:
from pathlib import Path
try:
    import tomllib
except ModuleNotFoundError:  # for Python < 3.11
    import tomli as tomllib
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.preprocessing import ICA

### File overview

The chapter relies on the annotated FIF derivatives created by `02_artefact_annotation.ipynb`. With the default config, the inputs are:

~~~text
<ROOT>/derivatives/mne-preprocessing/sub-06/ses-20260526/meg/sub-06_ses-20260526_task-movdot_run-01_desc-annotated_meg.fif
<ROOT>/derivatives/mne-preprocessing/sub-06/ses-20260526/meg/sub-06_ses-20260526_task-movdot_run-02_desc-annotated_meg.fif
<ROOT>/derivatives/mne-preprocessing/sub-06/ses-20260526/meg/sub-06_ses-20260526_task-movdot_run-03_desc-annotated_meg.fif
~~~

and generates one subject/session ICA model plus one cleaned derivative per configured run:

~~~text
<ROOT>/derivatives/mne-preprocessing/sub-06/ses-20260526/meg/sub-06_ses-20260526_task-movdot_desc-ica_ica.fif
<ROOT>/derivatives/mne-preprocessing/sub-06/ses-20260526/meg/sub-06_ses-20260526_task-movdot_run-01_desc-ica_meg.fif
<ROOT>/derivatives/mne-preprocessing/sub-06/ses-20260526/meg/sub-06_ses-20260526_task-movdot_run-02_desc-ica_meg.fif
<ROOT>/derivatives/mne-preprocessing/sub-06/ses-20260526/meg/sub-06_ses-20260526_task-movdot_run-03_desc-ica_meg.fif
~~~

To process participant 07, change `subject`, `session`, and `runs` in `config.toml`.

Read `config.toml`, select the configured BIDS subject/session/runs, and build the annotated inputs, ICA model file, and cleaned output file names. The ICA is fitted across the configured runs and then applied run by run.

In [3]:
script_dir = Path('__file__').resolve().parent
config_file = Path('config.toml')
if not config_file.exists():
    for parent in Path.cwd().resolve().parents:
        candidate = parent / 'config.toml'
        if candidate.exists():
            config_file = candidate
            break
if not config_file.exists():
    raise FileNotFoundError('Could not find config.toml. Start Jupyter from the pipeline root or edit config_file.')

with config_file.open("rb") as fid:
    config = tomllib.load(fid)

root = Path(config["paths"]["root"])
if not root.is_absolute():
    root = (config_file.parent / root).resolve()

subject = str(config["dataset"]["subject"]).zfill(2)
session = str(config["dataset"]["session"])
task = str(config["dataset"]["task"])
run_labels = [f"{int(run):02d}" for run in config["dataset"]["runs"]]

bids_subject = f"sub-{subject}"
bids_session = f"ses-{session}"
bids_prefix = f"{bids_subject}_{bids_session}_task-{task}"

deriv_root = root / config["paths"]["derivatives"]
deriv_meg_dir = deriv_root / bids_subject / bids_session / "meg"
report_dir = deriv_root / bids_subject / bids_session / "reports"
deriv_meg_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)

report_file = report_dir / "03_ica.html"
report = mne.Report(title=f"03 ICA - {bids_subject} {bids_session}")

ann_files = [
    deriv_meg_dir / f"{bids_prefix}_run-{run}_desc-annotated_meg.fif"
    for run in run_labels
]
missing_ann_files = [path for path in ann_files if not path.is_file()]
if missing_ann_files:
    missing = "\n  ".join(str(path) for path in missing_ann_files)
    raise FileNotFoundError(
        "Missing configured annotated files. Run 02_artefact_annotation.py first.\n  "
        f"{missing}"
    )

ica_model_file = deriv_meg_dir / f"{bids_prefix}_desc-ica_ica.fif"
ica_output_files = {
    ann_file: deriv_meg_dir / ann_file.name.replace(
        "_desc-annotated_meg.fif", "_desc-ica_meg.fif"
    )
    for ann_file in ann_files
}

print("Config file:")
print("  ", config_file)
print("Pipeline root:")
print("  ", root)
print("Subject/session/task/runs:")
print("  ", bids_subject, bids_session, task, run_labels)

print("Input annotated files:")
for path in ann_files:
    print("  ", path)

print("\nICA model file:")
print("  ", ica_model_file)

print("\nCleaned output files:")
for path in ica_output_files.values():
    print("  ", path)

print("\nReport file:")
print("  ", report_file)

Embedding : jquery-3.6.0.min.js
Embedding : bootstrap.bundle.min.js
Embedding : bootstrap.min.css
Embedding : bootstrap-table/bootstrap-table.min.js
Embedding : bootstrap-table/bootstrap-table.min.css
Embedding : bootstrap-table/bootstrap-table-copy-rows.min.js
Embedding : bootstrap-table/bootstrap-table-export.min.js
Embedding : bootstrap-table/tableExport.min.js
Embedding : bootstrap-icons/bootstrap-icons.mne.min.css
Embedding : highlightjs/highlight.min.js
Embedding : highlightjs/atom-one-dark-reasonable.min.css
Config file:
   /Users/goal0312/Desktop/thesis/data/config.toml
Pipeline root:
   /Users/goal0312/Desktop/thesis/data
Subject/session/task/runs:
   sub-01 ses-01 IceSkating ['01']
Input annotated files:
   /Users/goal0312/Desktop/thesis/data/derivatives/mne-preprocessing/sub-01/ses-01/meg/sub-01_ses-01_task-IceSkating_run-01_desc-annotated_meg.fif

ICA model file:
   /Users/goal0312/Desktop/thesis/data/derivatives/mne-preprocessing/sub-01/ses-01/meg/sub-01_ses-01_task-IceSka

Define the eye-tracker MISC channels that were used in the artefact annotation notebook. These channels are not used for automatic ICA scoring here; they are plotted as synchronized timing information to help interpret components.

In [ ]:
mne.compute_rank(manual_raw, 

In [4]:
left_eye = config['eye_tracker']['left']
right_eye = config['eye_tracker']['right']
left_eye_channels = {
    'x': left_eye[0],
    'y': left_eye[1],
    'pupil_or_validity': left_eye[2],
}
right_eye_channels = {
    'x': right_eye[0],
    'y': right_eye[1],
    'pupil_or_validity': right_eye[2],
}

eye_tracker_channels = [
    left_eye_channels['x'],
    left_eye_channels['y'],
    left_eye_channels['pupil_or_validity'],
    right_eye_channels['x'],
    right_eye_channels['y'],
    right_eye_channels['pupil_or_validity'],
]
print(eye_tracker_channels)

['MISC001', 'MISC002', 'MISC003', 'MISC004', 'MISC005', 'MISC006']


### Resampling and filtering of the raw data

In order to perform the ICA decomposition efficiently, we first prepare a copy of each annotated run. These copies contain MEG channels only, are down-sampled to 200 Hz, and are band-pass filtered from 1 to 40 Hz.

The prepared runs are then concatenated for ICA fitting. This is the first preprocessing step where concatenation is useful: ICA needs enough data to estimate stable components, and one subject-level decomposition should be applied consistently to all runs.

**Question 1:** Why concatenate runs for ICA but not for MaxFilter or artefact annotation?

**Answer:** MaxFilter depends on run-specific acquisition metadata and head-position information, and artefact annotation depends on run-specific timing. ICA, in contrast, estimates a subject-level component basis from the cleaned continuous signal. Concatenating prepared MEG-only copies gives ICA more data while preserving the original annotated FIF files for later application.

In [5]:
fit_raws = []
fit_summaries = []
for ann_file in ann_files:
    print(f"\nPreparing {ann_file.name}")
    raw = mne.io.read_raw_fif(ann_file, preload=True, verbose=True)

    missing_eye_channels = [ch for ch in eye_tracker_channels if ch not in raw.ch_names]
    if missing_eye_channels:
        print("Missing eye-tracker channels in this run:", missing_eye_channels)

    raw_fit = raw.copy().pick("meg")
    raw_fit.resample(config["ica"]["resample_sfreq"])
    raw_fit.filter(config["ica"]["l_freq"], config["ica"]["h_freq"])

    fit_summaries.append({
        "file": ann_file.name,
        "n_channels": raw_fit.info["nchan"],
        "n_samples": raw_fit.n_times,
        "sfreq": raw_fit.info["sfreq"],
        "bad_channels": list(raw_fit.info["bads"]),
    })
    fit_raws.append(raw_fit)

raw_resmpl_all = mne.concatenate_raws(fit_raws, on_mismatch="warn")
print("\nPrepared data for ICA:")
print(raw_resmpl_all)
print(fit_summaries)


Preparing sub-01_ses-01_task-IceSkating_run-01_desc-annotated_meg.fif
Opening raw data file /Users/goal0312/Desktop/thesis/data/derivatives/mne-preprocessing/sub-01/ses-01/meg/sub-01_ses-01_task-IceSkating_run-01_desc-annotated_meg.fif...
    Range : 20000 ... 751999 =     20.000 ...   751.999 secs
Ready.
Reading 0 ... 731999  =      0.000 ...   731.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 661 samples (3.305 s)


Prepared data for ICA:
<Raw | sub-01_ses-01_task-IceSkating_run-01_desc-a

### Applying the ICA algorithm

We apply the FastICA algorithm, which is well established and widely used. To keep this example aligned with the FLUX notebook and the DAD preprocessing defaults, we use 30 components and a fixed random seed.

Annotations are used during fitting, so data segments marked as bad in the previous notebook do not dominate the ICA decomposition.

In [6]:
## given the sensor recordings, figure out what the original, unmixed sources were.
## source
## s_1=3 (say, a blink signal)
## s_2​=−1 (say, brain activity)

## Sensor 1 reading: x1=0.8 s_1 + 0.3 s_2
## Sensor 2 reading: x2=0.2 s_1 + 0.9 s_2
## x(t) = A.s(t) => signal = weight_sensor s x feature_sensor A
## ICA estimates both A & s(t) from x(t)
ica = ICA(
    method='fastica',
    random_state=config['ica']['random_state'],
    n_components=config['ica']['n_components'],
    max_iter='auto',
    verbose=True,
)
ica.fit(raw_resmpl_all, reject_by_annotation=True, verbose=True)

ica.save(ica_model_file, overwrite=True)
print('Saved ICA model:', ica_model_file)

Fitting ICA to data using 306 channels (please be patient, this may take a while)
Omitting 2270 of 146400 (1.55%) samples, retaining 144130 (98.45%) samples.
Selecting by number: 30 components
Fitting ICA took 19.4s.
Overwriting existing file.
Writing ICA solution to /Users/goal0312/Desktop/thesis/data/derivatives/mne-preprocessing/sub-01/ses-01/meg/sub-01_ses-01_task-IceSkating_desc-ica_ica.fif...
Overwriting existing file.
Saved ICA model: /Users/goal0312/Desktop/thesis/data/derivatives/mne-preprocessing/sub-01/ses-01/meg/sub-01_ses-01_task-IceSkating_desc-ica_ica.fif


### Identifying the ICA components reflecting the artefacts

Plot the time courses of the ICA components. Use the left/right arrow keys to scroll through time and the up/down arrow keys to scroll through components.

The annotated eye-tracker intervals from the previous notebook are useful for interpretation. Components that consistently peak during `BAD_blink` or `BAD_eye_movement` intervals are candidates for exclusion. Components with spatial patterns typical of broad frontal eye-related activity, or clear high-frequency muscle-like activity, should be inspected carefully before being excluded.

In [ ]:
%matplotlib inline
ica.plot_sources(raw_resmpl_all, title='ICA components')

Creating RawArray with float64 data, n_channels=30, n_times=146400
    Range : 4000 ... 150399 =     20.000 ...   751.995 secs
Ready.
Using qt as 2D backend.


shibokensupport/signature/parser.py:270: RuntimeWarning: pyside_type_init:_resolve_value

        UNRECOGNIZED:   'PySide6.QtWidgets.QWidget'
        OFFENDING LINE: '1:PySide6.QtWidgets.QFileDialog(self,parent:PySide6.QtWidgets.QWidget,f:PySide6.QtCore.Qt.WindowType)'
        
shibokensupport/signature/parser.py:270: RuntimeWarning: pyside_type_init:_resolve_value

        UNRECOGNIZED:   'PySide6.QtCore.Qt.WindowType'
        OFFENDING LINE: '1:PySide6.QtWidgets.QFileDialog(self,parent:PySide6.QtWidgets.QWidget,f:PySide6.QtCore.Qt.WindowType)'
        
shibokensupport/signature/parser.py:270: RuntimeWarning: pyside_type_init:_resolve_value

        UNRECOGNIZED:   'PySide6.QtWidgets.QWidget'
        OFFENDING LINE: '0:PySide6.QtWidgets.QFileDialog(self,parent:PySide6.QtWidgets.QWidget=nullptr,caption:QString=QString(),directory:QString=QString(),filter:QString=QString())'
        
shibokensupport/signature/parser.py:270: RuntimeWarning: pyside_type_init:_resolve_value

        UNRECO

<mne_qt_browser._pg_figure.MNEQtBrowser(0x7f958f9afe40) at 0x188462040>

: 

Plot the component topographies.

**Question 2:** What kind of component should be removed?

**Answer:** Remove components only when their time course, topography, and timing relative to the eye-tracker or muscle annotations make a coherent artefact interpretation. Do not remove a component only because it has a large amplitude or an interesting topography; it may contain brain signal.

In [ ]:
%matplotlib inline
ica.plot_components()

Plot the eye-tracker MISC channels and the existing artefact annotations for the first run. This is not an automatic scoring step; it is a visual guide for interpreting the ICA component time courses.

In [ ]:
def robust_zscore(values):
    """Return a robust z-score using the median absolute deviation."""
    values = np.asarray(values, dtype=float)
    median = np.nanmedian(values)
    mad = np.nanmedian(np.abs(values - median))
    scale = 1.4826 * mad
    if not np.isfinite(scale) or scale == 0:
        scale = np.nanstd(values)
    if not np.isfinite(scale) or scale == 0:
        return np.zeros_like(values)
    return (values - median) / scale


def annotation_mask(raw, descriptions):
    """Return a boolean mask for annotations whose description is selected."""
    descriptions = set(descriptions)
    mask = np.zeros(raw.n_times, dtype=bool)
    sfreq = raw.info['sfreq']
    for annotation in raw.annotations:
        if annotation['description'] not in descriptions:
            continue
        start = max(0, int(round(annotation['onset'] * sfreq)) - raw.first_samp)
        stop = min(raw.n_times, int(round((annotation['onset'] + annotation['duration']) * sfreq)) - raw.first_samp)
        if stop > start:
            mask[start:stop] = True
    return mask


def plot_eye_tracker_with_annotations(raw, title, start_sec=0, duration_sec=60):
    """Plot eye-tracker traces and eye-related annotation masks."""
    available = [ch for ch in eye_tracker_channels if ch in raw.ch_names]
    if not available:
        raise RuntimeError('No configured eye-tracker MISC channels found in this raw file.')

    sfreq = raw.info['sfreq']
    start = int(start_sec * sfreq)
    stop = min(raw.n_times, int((start_sec + duration_sec) * sfreq))
    times = raw.times[start:stop]
    data = raw.get_data(picks=available)[:, start:stop]
    blink_mask = annotation_mask(raw, ['BAD_blink'])[start:stop]
    movement_mask = annotation_mask(raw, ['BAD_eye_movement'])[start:stop]

    fig, axes = plt.subplots(len(available) + 1, 1, figsize=(14, 11), sharex=True)
    for ax, channel, values in zip(axes[:-1], available, data, strict=True):
        ax.plot(times, robust_zscore(values), linewidth=0.5)
        ax.set_ylabel(channel)
    axes[-1].plot(times, blink_mask.astype(float), label='BAD_blink')
    axes[-1].plot(times, movement_mask.astype(float), label='BAD_eye_movement')
    axes[-1].set_ylim(-0.1, 1.1)
    axes[-1].set_ylabel('mask')
    axes[-1].set_xlabel('time (s)')
    axes[-1].legend(loc='upper right')
    fig.suptitle(title)
    fig.tight_layout()
    return fig

first_ann_file = ann_files[0]
raw_review = mne.io.read_raw_fif(first_ann_file, preload=True, verbose=True)
plot_eye_tracker_with_annotations(
    raw_review,
    f'Eye-tracker annotation guide: {first_ann_file.name}',
    start_sec=0,
    duration_sec=60,
)

You can also plot properties for a small set of candidate components. Edit `candidate_components` after inspecting the source traces and topographies.

In [ ]:
candidate_components = config['ica']['candidate_components']
ica.plot_properties(raw_resmpl_all, picks=candidate_components)

Set the components to exclude.

Leave `ica.exclude_components` empty in `config.toml` until the ICA components have been reviewed. After inspecting the component time courses, topographies, and relation to eye-tracker annotations, put the selected component indices in that config list and rerun this cell before applying ICA.

In [ ]:
ica.exclude = config['ica']['exclude_components']
print('Components marked for exclusion:', ica.exclude)

## Applying ICA to all runs

Now apply the selected ICA exclusions to each original annotated run and save the cleaned files. The ICA decomposition was fitted on concatenated prepared data, but the correction is applied back to each annotated run separately. This preserves the run structure and the eye-tracker MISC channels.

In [ ]:
cleaned_runs = []
for ann_file in ann_files:
    print(f'\nApplying ICA to {ann_file.name}')
    raw_ica = mne.io.read_raw_fif(ann_file, preload=True, verbose=True)
    ica.apply(raw_ica)

    output_file = ica_output_files[ann_file]
    raw_ica.save(output_file, overwrite=True)
    cleaned_runs.append(raw_ica)
    print('Saved', output_file)

## Plotting the data to check the artefact reduction

We will examine a few frontal magnetometers from the first run to demonstrate the artefact reduction. The exact channels can be changed depending on the participant and artefact pattern.

In [ ]:
raw_before = raw_review
raw_after = cleaned_runs[0]

frontal_candidates = ['MEG0311', 'MEG0121', 'MEG1211', 'MEG1411']
chs = [ch for ch in frontal_candidates if ch in raw_before.ch_names]
if not chs:
    chs = raw_before.copy().pick('mag').ch_names[:4]
chan_idxs = [raw_before.ch_names.index(ch) for ch in chs]
print('Channels for before/after inspection:', chs)

Plot the data before the ICA projections were applied. Use the arrow buttons to scroll through time.

In [ ]:
%matplotlib inline
raw_before.plot(order=chan_idxs, duration=5)

Then plot the same channels after the ICA projections were applied.

**Question 3:** How do we know whether the ICA correction was successful?

**Answer:** The artefact should be reduced in the cleaned data while the background MEG signal remains plausible. If the cleaned data look flattened, distorted, or missing broad periods of signal, the excluded components should be reconsidered.

In [ ]:
%matplotlib inline
raw_after.plot(order=chan_idxs, duration=5)

## Preregistration and publication

Example text:

"The data were down-sampled to 200 Hz prior to Independent Component Analysis (ICA) and band-pass filtered at 1-40 Hz. FastICA was applied as implemented in MNE-Python, using 30 components and a fixed random seed of 97. ICA was fitted on concatenated MEG-only copies of all annotated acquisition runs, excluding previously annotated bad segments from the fitting procedure. Components reflecting artefacts were selected by visual inspection of component time courses, topographies, and their timing relative to eye-tracker-derived artefact annotations. The selected components were then removed from each annotated run separately."

## References

Hyvärinen A, Oja E. (2000). Independent component analysis: algorithms and applications. *Neural Networks*, 13(4-5), 411-430. doi: 10.1016/S0893-6080(00)00026-5.